# FlashRank Reranker

FlashRank는 **초경량이면서 초고속**으로 작동하는 오픈소스 Reranker 라이브러리입니다. 로컬에서 빠르게 재정렬이 필요한 경우에 이상적인 선택입니다.

## 학습 목표

- FlashRank의 특징과 장점 이해
- 경량 Reranker 모델 활용법 습득
- LangChain FlashrankRerank 통합 구현
- 속도 중심 Reranker 선택 전략 학습

## FlashRank의 특징

- ✅ **초고속**: 가장 빠른 재정렬 속도
- ✅ **초경량**: 최소 메모리 사용
- ✅ **무료**: 완전 오픈소스 (로컬 실행)
- ✅ **간편 설치**: 단순한 의존성
- ✅ **Production Ready**: 안정적인 성능

> 🎯 **강의 포인트**: FlashRank는 Cross-Encoder보다 훨씬 빠르고, 클라우드 API 비용도 없습니다. API 키 없이 바로 시작할 수 있어 학습과 빠른 프로토타이핑에 최적입니다.

> 🔑 **핵심 개념**: FlashRank는 TinyBERT 계열의 경량 모델을 사용합니다. Cross-Encoder 수준의 정확도는 아니지만, 속도 면에서 압도적입니다. "충분히 좋은 품질을 매우 빠르게" 원하는 상황에 적합해요.

> ⚠️ **자주 하는 실수**: `FlashrankRerank.model_rebuild()`를 호출하지 않으면 Pydantic v2 환경에서 오류가 발생합니다. import 직후 반드시 `model_rebuild()`를 먼저 실행하세요.

## 1. 환경 설정

### 1.1 FlashRank 설치

```bash
pip install flashrank
```

### 1.2 FlashRank 모델

- `ms-marco-MultiBERT-L-12`: 다국어 지원 (권장)
- `ms-marco-MiniLM-L-12-v2`: 영어 특화, 높은 정확도
- `ms-marco-TinyBERT-L-2-v2`: 최고 속도, 경량

**특징**:
- API 키 불필요 (완전 로컬)
- 자동 모델 다운로드
- CPU에서도 빠른 성능


In [1]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

# 필요한 라이브러리 import
from dotenv import load_dotenv
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import FlashrankRerank
from flashrank import Ranker

# 모델 재빌드 (flashrank의 Ranker 등록 필요)
FlashrankRerank.model_rebuild()

# 환경 변수 로드
load_dotenv()

True

## 2. 문서 출력 도우미 함수


In [2]:
def pretty_print_docs(docs, show_scores=True):
    """검색된 문서를 보기 좋게 출력하는 함수"""
    print(f"\n{'=' * 100}")
    print(f"총 {len(docs)}개 문서 검색됨")
    print(f"{'=' * 100}\n")
    
    for i, doc in enumerate(docs, 1):
        print(f"[문서 {i}]")
        
        # FlashRank는 relevance_score를 metadata에 추가
        if show_scores and 'relevance_score' in doc.metadata:
            score = doc.metadata['relevance_score']
            print(f"관련성 점수: {score:.4f}")
        
        # 문서 ID 표시 (있는 경우)
        if 'id' in doc.metadata:
            print(f"문서 ID: {doc.metadata['id']}")
        
        print(f"내용: {doc.page_content}")
        print(f"{'─' * 100}\n")


## 3. 기본 검색 시스템 구축


In [3]:
# 1단계: 문서 로드
documents = TextLoader("./data/appendix-keywords.txt").load()

print(f"✅ 문서 로드 완료")
print(f"   - 문서 수: {len(documents)}")


✅ 문서 로드 완료
   - 문서 수: 1


In [4]:
# 2단계: 텍스트 분할 및 ID 추가
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

texts = text_splitter.split_documents(documents)

# 각 청크에 고유 ID 추가 (FlashRank는 ID를 사용)
for idx, text in enumerate(texts):
    text.metadata["id"] = idx

print(f"✅ 문서 분할 완료")
print(f"   - 청크 수: {len(texts)}")
print(f"   - 각 청크에 ID 추가 완료")


✅ 문서 분할 완료
   - 청크 수: 15
   - 각 청크에 ID 추가 완료


In [5]:
# 3단계: 벡터스토어 생성
vectorstore = FAISS.from_documents(texts, OpenAIEmbeddings())
base_retriever = vectorstore.as_retriever(
    search_kwargs={"k": 10}  # 초기 검색: 10개 문서
)

print("✅ 벡터스토어 생성 완료")
print(f"   - 임베딩: OpenAI")
print(f"   - 초기 검색 수: 10개")


✅ 벡터스토어 생성 완료
   - 임베딩: OpenAI
   - 초기 검색 수: 10개


In [6]:
# 기본 검색 테스트
query = "프로그래밍에서 코드를 분석하는 도구는?"

print(f"\n🔍 검색 쿼리: {query}\n")

# Reranker 적용 전 기본 검색
docs_before = base_retriever.invoke(query)

print("📊 [Reranker 적용 전] 벡터 유사도 기반 검색 결과:")
print(f"\n상위 3개 문서 ID: {[doc.metadata.get('id', 'N/A') for doc in docs_before[:3]]}")
pretty_print_docs(docs_before[:3], show_scores=False)



🔍 검색 쿼리: 프로그래밍에서 코드를 분석하는 도구는?

📊 [Reranker 적용 전] 벡터 유사도 기반 검색 결과:

상위 3개 문서 ID: [8, 11, 7]

총 3개 문서 검색됨

[문서 1]
문서 ID: 8
내용: Parser

정의: 파서는 주어진 데이터(문자열, 파일 등)를 분석하여 구조화된 형태로 변환하는 도구입니다. 이는 프로그래밍 언어의 구문 분석이나 파일 데이터 처리에 사용됩니다.
예시: HTML 문서를 구문 분석하여 웹 페이지의 DOM 구조를 생성하는 것은 파싱의 한 예입니다.
연관키워드: 구문 분석, 컴파일러, 데이터 처리

TF-IDF (Term Frequency-Inverse Document Frequency)

정의: TF-IDF는 문서 내에서 단어의 중요도를 평가하는 데 사용되는 통계적 척도입니다. 이는 문서 내 단어의 빈도와 전체 문서 집합에서 그 단어의 희소성을 고려합니다.
예시: 많은 문서에서 자주 등장하지 않는 단어는 높은 TF-IDF 값을 가집니다.
연관키워드: 자연어 처리, 정보 검색, 데이터 마이닝

Deep Learning
────────────────────────────────────────────────────────────────────────────────────────────────────

[문서 2]
문서 ID: 11
내용: 판다스 (Pandas)

정의: 판다스는 파이썬 프로그래밍 언어를 위한 데이터 분석 및 조작 도구를 제공하는 라이브러리입니다. 이는 데이터 분석 작업을 효율적으로 수행할 수 있게 합니다.
예시: 판다스를 사용하여 CSV 파일을 읽고, 데이터를 정제하며, 다양한 분석을 수행할 수 있습니다.
연관키워드: 데이터 분석, 파이썬, 데이터 처리

GPT (Generative Pretrained Transformer)

정의: GPT는 대규모의 데이터셋으로 사전 훈련된 생성적 언어 모델로, 다양한 텍스트 기반 작업에 활용됩니다. 이는 입력된 텍스트에 기반하여 자연스러운 언어를 생성할 수 

## 4. FlashRank Reranker 적용

FlashRank를 사용하여 빠르게 재정렬합니다.


In [7]:
# 1단계: FlashRank Reranker 초기화
compressor = FlashrankRerank(
    model="ms-marco-MultiBERT-L-12",  # 다국어 지원 모델
    top_n=3  # 최종 반환: 상위 3개 문서
)

# 2단계: 압축 검색기 생성
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=base_retriever
)

print("✅ FlashRank Reranker 설정 완료")
print(f"   - 모델: ms-marco-MultiBERT-L-12")
print(f"   - 실행 환경: 로컬 (CPU)")
print(f"   - 초기 검색: 10개 → 최종 반환: 3개")
print(f"   - API 키: 불필요 (완전 로컬)")


✅ FlashRank Reranker 설정 완료
   - 모델: ms-marco-MultiBERT-L-12
   - 실행 환경: 로컬 (CPU)
   - 초기 검색: 10개 → 최종 반환: 3개
   - API 키: 불필요 (완전 로컬)


In [8]:
# Reranker 적용 검색
import time

print(f"🔍 검색 쿼리: {query}\n")

# 속도 측정을 위한 시간 기록
start_time = time.time()

docs_after = compression_retriever.invoke(query)

elapsed_time = time.time() - start_time

print(f"⚡ FlashRank 실행 시간: {elapsed_time:.3f}초\n")
print("🎯 [Reranker 적용 후] FlashRank 재정렬 결과:")
print(f"\n재정렬된 문서 ID: {[doc.metadata.get('id', 'N/A') for doc in docs_after]}")
pretty_print_docs(docs_after, show_scores=True)


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


🔍 검색 쿼리: 프로그래밍에서 코드를 분석하는 도구는?

⚡ FlashRank 실행 시간: 0.930초

🎯 [Reranker 적용 후] FlashRank 재정렬 결과:

재정렬된 문서 ID: [3, 11, 1]

총 3개 문서 검색됨

[문서 1]
관련성 점수: 0.9995
문서 ID: 3
내용: JSON

정의: JSON(JavaScript Object Notation)은 경량의 데이터 교환 형식으로, 사람과 기계 모두에게 읽기 쉬운 텍스트를 사용하여 데이터 객체를 표현합니다.
예시: {"이름": "홍길동", "나이": 30, "직업": "개발자"}는 JSON 형식의 데이터입니다.
연관키워드: 데이터 교환, 웹 개발, API

Transformer

정의: 트랜스포머는 자연어 처리에서 사용되는 딥러닝 모델의 한 유형으로, 주로 번역, 요약, 텍스트 생성 등에 사용됩니다. 이는 Attention 메커니즘을 기반으로 합니다.
예시: 구글 번역기는 트랜스포머 모델을 사용하여 다양한 언어 간의 번역을 수행합니다.
연관키워드: 딥러닝, 자연어 처리, Attention

HuggingFace
────────────────────────────────────────────────────────────────────────────────────────────────────

[문서 2]
관련성 점수: 0.9994
문서 ID: 11
내용: 판다스 (Pandas)

정의: 판다스는 파이썬 프로그래밍 언어를 위한 데이터 분석 및 조작 도구를 제공하는 라이브러리입니다. 이는 데이터 분석 작업을 효율적으로 수행할 수 있게 합니다.
예시: 판다스를 사용하여 CSV 파일을 읽고, 데이터를 정제하며, 다양한 분석을 수행할 수 있습니다.
연관키워드: 데이터 분석, 파이썬, 데이터 처리

GPT (Generative Pretrained Transformer)

정의: GPT는 대규모의 데이터셋으로 사전 훈련된 생성적 언어 모델로, 다양한 텍스트 기반 작업에 활용됩니다. 이는 입력된 텍스트

## 5. 결과 비교 및 분석


In [9]:
print("\n" + "=" * 100)
print("📊 FlashRank Reranker 효과 분석")
print("=" * 100)

print(f"\n[검색 쿼리]")
print(f"  {query}")

print(f"\n[Reranker 적용 전] - 상위 3개 (문서 ID)")
for i, doc in enumerate(docs_before[:3], 1):
    doc_id = doc.metadata.get('id', 'N/A')
    preview = doc.page_content.replace('\n', ' ')[:80]
    print(f"  {i}. [ID: {doc_id}] {preview}...")

print(f"\n[Reranker 적용 후] - 상위 3개 (문서 ID + 관련성 점수)")
for i, doc in enumerate(docs_after, 1):
    doc_id = doc.metadata.get('id', 'N/A')
    score = doc.metadata.get('relevance_score', 0)
    preview = doc.page_content.replace('\n', ' ')[:80]
    print(f"  {i}. [ID: {doc_id}, 점수: {score:.4f}] {preview}...")

print(f"\n⏱️  실행 시간: {elapsed_time:.3f}초 (매우 빠름)")

print("\n💡 FlashRank의 강점:")
print("  ✅ 초고속: 가장 빠른 재정렬 속도")
print("  ✅ 초경량: 최소 메모리 사용")
print("  ✅ 무료: API 비용 없음")
print("  ✅ 로컬 실행: 데이터 프라이버시 보장")



📊 FlashRank Reranker 효과 분석

[검색 쿼리]
  프로그래밍에서 코드를 분석하는 도구는?

[Reranker 적용 전] - 상위 3개 (문서 ID)
  1. [ID: 8] Parser  정의: 파서는 주어진 데이터(문자열, 파일 등)를 분석하여 구조화된 형태로 변환하는 도구입니다. 이는 프로그래밍 언어의 구문 분석...
  2. [ID: 11] 판다스 (Pandas)  정의: 판다스는 파이썬 프로그래밍 언어를 위한 데이터 분석 및 조작 도구를 제공하는 라이브러리입니다. 이는 데이터 분석...
  3. [ID: 7] Open Source  정의: 오픈 소스는 소스 코드가 공개되어 누구나 자유롭게 사용, 수정, 배포할 수 있는 소프트웨어를 의미합니다. 이는 협...

[Reranker 적용 후] - 상위 3개 (문서 ID + 관련성 점수)
  1. [ID: 3, 점수: 0.9995] JSON  정의: JSON(JavaScript Object Notation)은 경량의 데이터 교환 형식으로, 사람과 기계 모두에게 읽기 쉬운 텍...
  2. [ID: 11, 점수: 0.9994] 판다스 (Pandas)  정의: 판다스는 파이썬 프로그래밍 언어를 위한 데이터 분석 및 조작 도구를 제공하는 라이브러리입니다. 이는 데이터 분석...
  3. [ID: 1, 점수: 0.9994] Token  정의: 토큰은 텍스트를 더 작은 단위로 분할하는 것을 의미합니다. 이는 일반적으로 단어, 문장, 또는 구절일 수 있습니다. 예시: ...

⏱️  실행 시간: 0.930초 (매우 빠름)

💡 FlashRank의 강점:
  ✅ 초고속: 가장 빠른 재정렬 속도
  ✅ 초경량: 최소 메모리 사용
  ✅ 무료: API 비용 없음
  ✅ 로컬 실행: 데이터 프라이버시 보장


## 6. 다양한 쿼리 테스트


In [10]:
# 다양한 쿼리로 FlashRank 성능 검증
test_queries = [
    "인공신경망을 활용하여 복잡한 문제를 해결하는 기술은?",
    "웹 페이지의 중요도를 평가하여 순위를 매기는 알고리즘은?",
    "JSON과 비슷하게 경량 데이터를 교환하는 형식은?"
]

print("\n" + "=" * 100)
print("🧪 FlashRank 다양한 쿼리 테스트")
print("=" * 100)

for idx, test_query in enumerate(test_queries, 1):
    print(f"\n{'─' * 100}")
    print(f"[테스트 {idx}] 쿼리: {test_query}")
    print(f"{'─' * 100}")
    
    # 속도 측정
    start_time = time.time()
    reranked_docs = compression_retriever.invoke(test_query)
    elapsed = time.time() - start_time
    
    print(f"\n⚡ 실행 시간: {elapsed:.3f}초")
    print(f"✅ FlashRank가 선택한 상위 {len(reranked_docs)}개 문서:")
    
    for i, doc in enumerate(reranked_docs, 1):
        doc_id = doc.metadata.get('id', 'N/A')
        score = doc.metadata.get('relevance_score', 0)
        preview = doc.page_content.replace('\n', ' ')[:70]
        print(f"\n  [{i}] ID: {doc_id}, 점수: {score:.4f}")
        print(f"      {preview}...")


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"



🧪 FlashRank 다양한 쿼리 테스트

────────────────────────────────────────────────────────────────────────────────────────────────────
[테스트 1] 쿼리: 인공신경망을 활용하여 복잡한 문제를 해결하는 기술은?
────────────────────────────────────────────────────────────────────────────────────────────────────


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"



⚡ 실행 시간: 0.955초
✅ FlashRank가 선택한 상위 3개 문서:

  [1] ID: 3, 점수: 0.9993
      JSON  정의: JSON(JavaScript Object Notation)은 경량의 데이터 교환 형식으로, 사람과 기계 모두...

  [2] ID: 13, 점수: 0.9990
      Page Rank  정의: 페이지 랭크는 웹 페이지의 중요도를 평가하는 알고리즘으로, 주로 검색 엔진 결과의 순위를 결정하는 ...

  [3] ID: 6, 점수: 0.9990
      정의: LLM은 대규모의 텍스트 데이터로 훈련된 큰 규모의 언어 모델을 의미합니다. 이러한 모델은 다양한 자연어 이해 및 생성...

────────────────────────────────────────────────────────────────────────────────────────────────────
[테스트 2] 쿼리: 웹 페이지의 중요도를 평가하여 순위를 매기는 알고리즘은?
────────────────────────────────────────────────────────────────────────────────────────────────────


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"



⚡ 실행 시간: 0.967초
✅ FlashRank가 선택한 상위 3개 문서:

  [1] ID: 12, 점수: 0.9993
      InstructGPT  정의: InstructGPT는 사용자의 지시에 따라 특정한 작업을 수행하기 위해 최적화된 GPT 모델입...

  [2] ID: 11, 점수: 0.9991
      판다스 (Pandas)  정의: 판다스는 파이썬 프로그래밍 언어를 위한 데이터 분석 및 조작 도구를 제공하는 라이브러리입니다....

  [3] ID: 6, 점수: 0.9989
      정의: LLM은 대규모의 텍스트 데이터로 훈련된 큰 규모의 언어 모델을 의미합니다. 이러한 모델은 다양한 자연어 이해 및 생성...

────────────────────────────────────────────────────────────────────────────────────────────────────
[테스트 3] 쿼리: JSON과 비슷하게 경량 데이터를 교환하는 형식은?
────────────────────────────────────────────────────────────────────────────────────────────────────

⚡ 실행 시간: 0.940초
✅ FlashRank가 선택한 상위 3개 문서:

  [1] ID: 11, 점수: 0.9996
      판다스 (Pandas)  정의: 판다스는 파이썬 프로그래밍 언어를 위한 데이터 분석 및 조작 도구를 제공하는 라이브러리입니다....

  [2] ID: 5, 점수: 0.9996
      Crawling  정의: 크롤링은 자동화된 방식으로 웹 페이지를 방문하여 데이터를 수집하는 과정입니다. 이는 검색 엔진 최적화...

  [3] ID: 10, 점수: 0.9996
      DataFrame  정의: DataFrame은 행과 열로 이루어진 테이블 형태의 데이터 구조로, 주로 데이터 분석 및 처리에 ...


## 7. 핵심 정리

### 전체 Reranker 비교

| 특징 | FlashRank | Cross-Encoder | Jina | Cohere |
|------|:---:|:---:|:---:|:---:|
| 속도 | 최고 | 보통 | 빠름 | 빠름 |
| 정확도 | 보통 | 높음 | 높음 | 최고 |
| 비용 | 무료 | 무료 | $ | $$ |
| 토큰 길이 | 512 | 512 | 8K | 4K |
| 실행 환경 | 로컬 (CPU) | 로컬 (GPU 권장) | 클라우드 | 클라우드 |
| 추천 용도 | 빠른 프로토타입 | 개발/테스트 | 긴 문서 | 프로덕션 |

### 기본 사용법

```python
from langchain.retrievers.document_compressors import FlashrankRerank

compressor = FlashrankRerank(
    model="ms-marco-MultiBERT-L-12",
    top_n=3,
)
```

### 주요 파라미터

| 파라미터 | 설명 | 기본값 |
|---------|------|--------|
| `model` | FlashRank 모델명 | `ms-marco-MultiBERT-L-12` |
| `top_n` | 최종 반환 문서 수 | 3 |

> 💡 **실무 팁**: Reranker 4종 중 어떤 것을 선택할지 고민된다면, 먼저 FlashRank로 빠르게 구현하고 검색 품질을 측정한 뒤, 필요하다면 Cross-Encoder나 Cohere로 업그레이드하는 순서를 추천합니다.

> ⚠️ **자주 하는 실수**: FlashRank의 `metadata["id"]`는 문서 추적에 사용됩니다. 문서에 ID를 추가하지 않으면 재정렬 결과에서 어떤 원본 문서인지 추적하기 어려워요. 항상 문서 ID를 메타데이터에 포함시키세요.